In [2]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display
from pathlib import Path

from src.data_interface import loader, writer
from src.data_interface.staging import stage_fields, field_status
from src.pipeline import run
from src.pipeline.context import RunContext
from src.calculators import travel_time_index
from src.utilities.visualize import kepler_visualize

import json

from src.config import CONFIG_DIR, LIBRARY_PATH
from src.data_interface.validation import _field_key, load_schema, validate_table
from src.utilities import get_domain
from src.utilities.naming import snake_case

context = RunContext("TestScenario")

ModuleNotFoundError: No module named 'keplergl'

In [2]:
# Change to the local shortcut to the SharePoint location for this project
# It's tricky to directly link to the SharePoint because of authentication issues

SHAREPOINT_SHORTCUT = Path("C:/Users/lij11/OneDrive/OneDrive - AECOM/H-GAC ATSPT (Subs) - General").resolve()
DATA_FOLDER = SHAREPOINT_SHORTCUT / "400_Technical" / "431_Task 1 Data Hub + Collection"
CONSTANTS_DATA_FOLDER = DATA_FOLDER / "4_Constants_Tables"


In [3]:
tazs = (
    gpd.read_parquet(CONSTANTS_DATA_FOLDER / "taz.parquet")
    .rename(columns={"taz5808": "taz_id"})
)
tazs['taz_id'] = tazs['taz_id'].astype(str)
tazs = tazs.set_index("taz_id")
corridors = (
    gpd.read_parquet(CONSTANTS_DATA_FOLDER / "corridors.parquet")
    .rename(columns={"length_mi": "length_miles"})
    .drop(columns="taz_count")
    .set_index("corridor_id")
)
subregions = gpd.read_parquet(CONSTANTS_DATA_FOLDER / "subregions.parquet").set_index(
    "subregion_id"
)
# County/place IDs contain FIPS; precincts and unincorporated areas have no separate code.
subregions["fips"] = (
    subregions.index.to_series().str.extract(r"^(?:COUNTY|CITY)_(\d+)$", expand=False).astype("Int64")
)
cells = gpd.read_parquet(CONSTANTS_DATA_FOLDER / "cells.parquet")[[
    'cell_id',
    'area_sqft',
    'base_year_contains_retail',
    'base_year_contains_recreation',
    'base_year_contains_healthcare',
    'flood_exposure',
    'taz_id',
    'sub_region_id',
    'corridor_id',
]].rename(columns={'taz_id': 'taz_ids', 'sub_region_id': 'subregion_ids', 'corridor_id': 'corridor_ids'})
cells['flood_exposure'] = cells['flood_exposure'] > 0
cells = cells.set_index("cell_id")
corridors["road_category"] = corridors["road_category"].fillna("Uncategorized")

### Base-year land use and scenario placeholders

The Montgomery County outputs cover a subset of Cells and TAZs. Use their base-year values for TestScenario and leave uncovered rows and unavailable fields null.

In [4]:
LAND_USE_OUTPUT = loader.DATA_DIR.parent / "_temp" / "land-use" / "data" / "output"
cell_land_use = pd.read_parquet(LAND_USE_OUTPUT / "cell_land_use.parquet").set_index("cell_id")
taz_demographics = pd.read_parquet(LAND_USE_OUTPUT / "taz_demographics.parquet").set_index("taz_id")
taz_demographics.index = taz_demographics.index.astype(str)

assert cell_land_use.index.is_unique and cell_land_use.index.isin(cells.index).all()
assert taz_demographics.index.is_unique and taz_demographics.index.isin(tazs.index).all()

# Align to all regional IDs; missing source rows stay null, including booleans.
cell_land_use["contains_retail"] = cell_land_use["contains_retail"].astype("boolean")
cell_land_use = cell_land_use.reindex(cells.index)
taz_demographics = taz_demographics.reindex(tazs.index)
for field in ("population", "households", "workers", "jobs", "imperviousness"):
    cells[f"base_year_{field}"] = cell_land_use[field]
for field in ("population", "workers", "jobs"):
    tazs[f"base_year_{field}"] = taz_demographics[field]

# These scenario attributes are not available in the base-year output.
cell_land_use["contains_transit"] = pd.Series(pd.NA, index=cells.index, dtype="boolean")
cell_land_use["e_vtol_compatability"] = pd.Series(pd.NA, index=cells.index, dtype="Float64")

### BaseHighwayLinks — source classifications and tentative facility mapping


In [5]:
base_highway_links = gpd.read_parquet(CONSTANTS_DATA_FOLDER / "base_highway_links.parquet")

In [6]:
base = base_highway_links.set_index("link_id").rename(columns={
    "distance_miles": "base_year_distance_miles",
    "roadway_class": "base_year_roadway_class",
}).copy()

# Use the supplied cross section and access labels, preserving missing values.
base["base_year_cross_section"] = base["base_year_cross_section"].replace({
    "Not divided": "Undivided",
})
base["base_year_area_access_type"] = base["base_year_area_access_type"].replace({
    "Rural Non-restricted": "Rural Unrestricted",
    "Urban Non-restricted": "Urban Unrestricted",
    "N/A": pd.NA,
})

# The source describes facilities without an NHS designation. The crosswalk is provisional.
facility_map = {
    "Rural Interstate": "Interstate",
    "Urban Interstate": "Interstate",
    "Rural Prin. Arterials": "Non Interstate NHS",
    "Urban Prin. Arterials": "Non Interstate NHS",
    "Rural other freeways": "Non Interstate NHS",
    "Urban other Freeways": "Non Interstate NHS",
    "Toll Roads": "Non Interstate NHS",
    "Rural Collectors": "Other",
    "Urban Collectors": "Other",
    "Rural Major Collectors": "Other",
    "Rural Other Arterials": "Other",
    "Urban Other Arterials": "Other",
    "Ramps (Freeway/Toll/Frontage)": "Other",
}
base["base_year_facility_type"] = base["base_year_facility_type"].replace(facility_map)


# The refreshed source supplies roadway class; normalize to the exported domain.
base["base_year_roadway_class"] = base["base_year_roadway_class"].replace({
    "FringeRural": "Fringe Rural",
    "L2+3_Suburban": "L2 L3 Suburban",
    "L4+5_Suburban": "L4 L5 Suburban",
    "UrbanCore": "Urban Core",
})


### BaseYearTDM

The current schema omits daily and period truck volumes. Staging drops those source columns, retains daily truck VMT, and adds nullable medium/heavy truck ratios when absent.


In [7]:
base_year_tdm = pd.read_parquet(CONSTANTS_DATA_FOLDER / "base_year_tdm.parquet")
base_year_tdm = base_year_tdm.set_index("link_id")

# Keep current schema fields; removed truck-volume columns are not exported.
base_year_tdm = stage_fields(base_year_tdm, "PreprocessedConstants", "BaseYearTDM")


### BikePedAccessibilityMatrix


In [8]:
bike_ped_accessibility = pd.read_parquet(CONSTANTS_DATA_FOLDER / "bike_ped_accessibility.parquet")
bike_ped = bike_ped_accessibility.set_index("cell_id")

### HistoricCrashes — retain both crash-type flags and derive severity


In [9]:
crashes = gpd.read_parquet(CONSTANTS_DATA_FOLDER / "crashes.parquet")
crashes = crashes.set_index("crash_point_id", verify_integrity=True).rename(columns={
    "crash_type_intersection": "is_intersection",
    "crash_type_ped_cyclist": "involves_pedestrian_cyclist",
}).copy()
crashes["severity"] = crashes.severity.str.extract(r"\(([KABCO])\)\s*$", expand=False)
# Keep independent flags, including overlaps and unknowns. KPI 96 classifies them.
crashes["is_intersection"] = crashes["is_intersection"].astype("boolean")
crashes["involves_pedestrian_cyclist"] = crashes["involves_pedestrian_cyclist"].astype("boolean")
crash_flag_overlap = int((crashes["involves_pedestrian_cyclist"] & crashes["is_intersection"]).sum())
print(f"HistoricCrashes: {len(crashes):,} source rows; {crash_flag_overlap:,} rows have both flags; both flags retained")


HistoricCrashes: 817,874 source rows; 6,747 rows have both crash-type flags; Type remains unresolved


### ScenarioLinks — stage the supplied future-network attributes


In [10]:
# Load the updated scenario source; sample projects are assigned after network alignment.
scenario_links = gpd.read_parquet(CONSTANTS_DATA_FOLDER / "scenario_links.parquet").set_index(
    "link_id", verify_integrity=True
)
assert scenario_links.index.notna().all(), "ScenarioLinks has a missing link ID"
# Normalize source labels to the same domains used for the base network.
scenario_links["cross_section_type"] = scenario_links["cross_section_type"].replace({
    "Not divided": "Undivided",
})
scenario_links["area_access_type"] = scenario_links["area_access_type"].replace({
    "Rural Non-restricted": "Rural Unrestricted",
    "Urban Non-restricted": "Urban Unrestricted",
    "N/A": pd.NA,
})
scenario_links["facility_type"] = scenario_links["facility_type"].replace(facility_map)
scenario_links["roadway_class"] = scenario_links["roadway_class"].replace({
    "FringeRural": "Fringe Rural",
    "L2+3_Suburban": "L2 L3 Suburban",
    "L4+5_Suburban": "L4 L5 Suburban",
    "UrbanCore": "Urban Core",
})
scenario_links["is_removed_from_base"] = False


In [11]:
inrix = pd.read_parquet(CONSTANTS_DATA_FOLDER / "inrix.parquet")
inrix["link_id"] = inrix["link_id"].astype("string")
inrix = inrix.set_index("link_id")


In [12]:
zone_accessibility = pd.read_parquet(CONSTANTS_DATA_FOLDER / "zone_accessibility.parquet")
zone_accessibility = zone_accessibility.rename(columns={"taz5808": "taz_id"})
zone_accessibility["taz_id"] = zone_accessibility["taz_id"].astype("string")
zone_accessibility = zone_accessibility.set_index("taz_id")
for field in ("drive_15_min_taz_ids", "drive_30_min_taz_ids", "transit_45_min_taz_ids"):
    zone_accessibility[field] = zone_accessibility[field].map(
        lambda ids: [str(value) for value in ids] if isinstance(ids, (list, tuple, np.ndarray)) else None
    )


In [13]:
joined = inrix.reset_index().merge(base_highway_links[['link_id']], how="left", on="link_id", indicator=True)
assert joined["_merge"].eq("both").all(), "INRIX has IDs absent from BaseHighwayLinks"


### NetworkOutput — future-year TDM and supplied scenario links

Read from `data/_temp/network_output.parquet`. TDM can represent either year: `BaseYearTDM` is the shared base-year product, while this scenario's `NetworkOutput` is future-year TDM. Keep missing truck ratios and period occupancies as null placeholders. The updated `scenario_links.parquet` supplies every network ID, including added-link geometry and attributes. Preserve missing source values and verify exact ID coverage before aligning rows. Scenario link-level outputs use the ScenarioLinks index.

NetworkOutput supplies medium- and heavy-truck percentages. The calibration step uses them with period traffic volumes to derive calibrated truck volumes. Capacity comes from ScenarioLinks.


In [14]:
network_output = gpd.read_parquet(loader.DATA_DIR / "_temp" / "network_output.parquet")

# Load the future network; export only schema fields below.
network = network_output.set_index("link_id", verify_integrity=True).copy()
assert network.index.notna().all(), "NetworkOutput has a missing link ID"
# Missing truck percentages and period occupancy stay unknown.
network = stage_fields(network, "ScenarioUpstream", "NetworkOutput")

# Require complete source coverage; preserve supplied geometry and attributes.
assert scenario_links.index.isin(network.index).all() and network.index.isin(scenario_links.index).all()
scenario_links = scenario_links.reindex(network.index)
scenario_links["is_added_to_base"] = ~scenario_links.index.isin(base.index)
new_links = scenario_links.index[scenario_links["is_added_to_base"]]
scenario_links["is_removed_from_base"] = False

display(pd.Series({"NetworkOutput rows": len(network), "ScenarioLinks rows": len(scenario_links),
                   "Source-supplied added links": len(new_links),
                   "Unmatched network IDs": int((~network.index.isin(scenario_links.index)).sum()),
                   "Scenario links without network output": int((~scenario_links.index.isin(network.index)).sum())}))
assert scenario_links.index.equals(network.index)


NetworkOutput rows                       47341
ScenarioLinks rows                       47341
Source-supplied added links               3800
Unmatched network IDs                        0
Scenario links without network output        0
dtype: int64

In [15]:
# Validate destination IDs as well as each origin TAZ key; null lists stay unknown.
taz_ids = set(tazs.index)
for field in zone_accessibility:
    assert all(set(ids) <= taz_ids for ids in zone_accessibility[field].dropna()), field


### Validate fields and joins


In [16]:
# Synthetic sample: three distinct links per project, selected within its sample cells.
# Store the 1:N foreign key on ScenarioLinks and retain the Projects ID type.
projects = loader.load(context, "Projects", category="ScenarioUpstream")
sample_links_per_project = 3
link_cells = base["cell_id"].reindex(scenario_links.index)
scenario_links["project_id"] = pd.Series(
    pd.NA, index=scenario_links.index,
    dtype=projects.index.to_series().convert_dtypes().dtype,
)
for project_id, project in projects.sort_index().iterrows():
    candidates = scenario_links.index[
        scenario_links["project_id"].isna()
        & ~scenario_links["is_removed_from_base"].fillna(False)
        & link_cells.isin(project["cell_ids"])
    ].sort_values()
    assert len(candidates) >= sample_links_per_project, f"Project {project_id}: insufficient unassigned links in its sample cells"
    scenario_links.loc[candidates[:sample_links_per_project], "project_id"] = project_id
assigned_counts = scenario_links["project_id"].value_counts().reindex(projects.index, fill_value=0)
assert assigned_counts.eq(sample_links_per_project).all()
assert scenario_links["project_id"].dropna().isin(projects.index).all()

tables = {"TAZs": tazs, "Corridors": corridors, "SubRegions": subregions, "Cells": cells,
          "BaseHighwayLinks": base, "BaseYearTDM": base_year_tdm,
          "BikePedAccessibilityMatrix": bike_ped, "HistoricCrashes": crashes,
          "ScenarioLinks": scenario_links, "INRIX": inrix,
          "ZoneAccessibilityMatrix": zone_accessibility,
          "NetworkOutput": network, "CellLandUse": cell_land_use,
          "TAZDemographics": taz_demographics,
          "Projects": projects}
schema = load_schema()
categories = {name: category for category, definitions in schema.items() for name in definitions}
# Restrict processed outputs before validation, status reporting, and writing.
for name, frame in tables.items():
    category = categories[name]
    if category != "PreprocessedConstants" and name != "ScenarioLinks" and schema[category][name]["PK"] == "LinkID":
        assert frame.index.isin(scenario_links.index).all(), f"{name} has IDs outside ScenarioLinks"
        frame = frame.reindex(scenario_links.index)
    tables[name] = stage_fields(frame, category, name)
for name in tables:
    validate_table(context, categories[name], name, tables=tables)
cell_ids = set(cells.index)
assert all(set(ids) <= cell_ids for column in bike_ped for ids in bike_ped[column])

### Write tables and compact field status

Status includes numeric missing percentages (including 100%), unresolved issues, and library derivation formulas where available. Existing Projects are included in the audit. Each sample project receives three distinct links within its sample cells, selected by sorted project and link IDs. These assignments are synthetic; `ScenarioLinks.project_id` stores the foreign key, and other links remain unassigned.


In [17]:
# Compact field status, including derivation references and unresolved provenance.
table_status = {}
for name, frame in tables.items():
    issues = {field: "tentative facility crosswalk; confirm NHS membership" for field in frame if field in {"base_year_facility_type", "facility_type"}}
    if name == "ScenarioLinks":
        issues["project_id"] = "synthetic sample: three links per project selected within its sample cell_ids; other links intentionally unassigned"
    if name in {"CellLandUse", "TAZDemographics"}:
        issues.update({field: "base-year sample used as scenario placeholder; scenario data pending"
                       for field in frame if frame[field].notna().any()})
    table_status[name] = field_status(frame, categories[name], name, issues=issues)
    scenario = context.scenario if categories[name] != "PreprocessedConstants" else None
    writer.write_table(context, name, frame, scenario=scenario)
(CONFIG_DIR / "_status.json").write_text(json.dumps(table_status, indent=2) + "\n", encoding="utf-8")


Out[17]: 14432


### Prepare the scenario and run KPI 1

The registered preparator writes the synthetic scenario network output before KPI calculation. It uses the selected NetworkOutput and preserves its link IDs.

Calibrated V/C follows the library formula by period: `100 * calibrated_volume / (ScenarioLinks.capacity / period_hours)`, with AM=3, PM=4, MD=6 and OV=11 hours. Missing or nonpositive capacity remains null. BaseYearTDM and NetworkOutput do not store capacity fields.


In [18]:
results = run.prepare(context, "calibrate_network_output")
assert all(result.status == "completed" for result in results), results
travel_time_index.calculate(context)

# Check the persisted rows, including links with no base record or usable KPI inputs.
stored_links = loader.load(context, "ScenarioLinks", category="ScenarioUpstream").index
assert stored_links.equals(network_output.set_index("link_id").index)
for category, table in [("ScenarioUpstream", "NetworkOutput"), ("KPIPreparation", "CalibratedNetworkOutput")]:
    assert loader.load(context, table, category=category).index.equals(stored_links), table
link_kpi = loader.load(context, travel_time_index.KPI_NAME, path="kpis", scale="Links")
assert link_kpi.index.equals(stored_links), travel_time_index.KPI_NAME

# Reload the preparator status so the summary includes CalibratedNetworkOutput.
table_status = json.loads((CONFIG_DIR / "_status.json").read_text(encoding="utf-8"))
display(pd.DataFrame({name: {category: len(fields) for category, fields in status.items()}
                      for name, status in table_status.items()}).T)


,completed_fields,fields_with_missing_values,problematic_fields
TAZs,3,3,0
Corridors,5,0,0
SubRegions,3,1,0
Cells,9,5,0
BaseHighwayLinks,5,20,4
BaseYearTDM,12,8,8
BikePedAccessibilityMatrix,6,0,0
HistoricCrashes,3,3,1
ScenarioLinks,6,11,3
INRIX,1,13,0
